# Smile Arc Classification — Fixed & Revised

This notebook fixes the following issues found in the original `SmileClassification_Revised_Runned.ipynb`:

1. **Augmentation never actually applied.** The original set `train_dataset.dataset.transform = ...` on a `ConcatDataset`, which silently does nothing. This version applies transforms per-sample in a custom `Dataset`, so train and test genuinely use different pipelines.
2. **`transforms` module was shadowed.** The original reassigned `transforms = transforms.Compose(...)`, breaking any later `transforms.Compose(...)` call in a fresh kernel. Renamed to `train_transform` / `eval_transform`.
3. **`base_transforms` was referenced before it was ever defined** in the original cell order. Removed — transforms are now passed explicitly into every function, no reliance on notebook execution order.
4. **No stratified split.** Replaced `random_split` with `sklearn.model_selection.train_test_split(..., stratify=labels)` so minority classes (reverse / straight non-consonant) are represented proportionally in train and test.
5. **No random seed.** Added global seeding for reproducibility.
6. **No class-imbalance handling.** Added inverse-frequency class weights passed to `CrossEntropyLoss`.
7. **No best-checkpoint selection.** Training now tracks test accuracy every epoch and keeps the best-performing weights instead of just the final epoch's.
8. **Confusion matrices and ROC curves were fabricated** (hand-tuned/estimated from precision-recall numbers in the original). This version computes both directly from real model predictions and predicted probabilities on the actual test set.
9. **Deprecated `pretrained=True` API** replaced with the current `weights=...` API for both ResNet50 and MobileNetV3-Large.

**Before running:** this notebook expects the same folder layout as the original (`data_root/smileArc`, `data_root/smileArcDSLR`, `data_root/smileArcMobile`, each containing one subfolder per class). If you're running this in Google Colab, mount your Drive first and set `DATA_ROOT` below to your actual path. This notebook has **not** been executed against your real dataset — do that yourself and check the printed class counts in the first experiment cell look right before trusting any results.


## 1. Imports & Reproducibility

In [ ]:
import os
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as tv_transforms
from PIL import Image, ImageFilter

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, f1_score
from sklearn.preprocessing import label_binarize

# --- Step 0: Download and Setup Dataset ---
# Downloading the zip from the user-provided URL
!gdown "https://drive.google.com/uc?id=1_VF4UkDuIsslIuS_SFc_z1GMSQh7hHIN" -O smile_photos.zip
!mkdir -p /content/data
!unzip -q smile_photos.zip -d /content/data

# --- Configuration ---
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
DATA_ROOT = "/content/data"
SOURCE_FOLDERS = ["smileArc", "smileArcDSLR", "smileArcMobile"]
CLASSES_4 = ["consonant", "not available", "reverse- non consonant", "straight- non consonant"]

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Downloading...
From (original): https://drive.google.com/uc?id=1_VF4UkDuIsslIuS_SFc_z1GMSQh7hHIN
From (redirected): https://drive.google.com/uc?id=1_VF4UkDuIsslIuS_SFc_z1GMSQh7hHIN&confirm=t&uuid=ce662906-5105-4d76-815e-caad019b68ee
To: /content/smile_photos.zip
100% 226M/226M [00:04<00:00, 54.4MB/s]
Using device: cuda


## 2. Data Configuration

If running in Colab, uncomment the drive-mount lines and set `DATA_ROOT` to your actual path.

In [19]:
# Global Configuration Constants
DATA_ROOT = "/content/data"
SOURCE_FOLDERS = ["smileArc", "smileArcDSLR", "smileArcMobile"]
CLASSES_4 = ["consonant", "not available", "reverse- non consonant", "straight- non consonant"]

# Consistent configuration for all cells
IMG_SIZE = 224
BATCH_SIZE = 32

## 3. Transforms

Note the renamed variables — nothing here shadows the `torchvision.transforms` module.

In [3]:
from torchvision import transforms as tv_transforms
from PIL import ImageFilter

# Configuration used for transforms
IMG_SIZE = 224

class GaussianBlur:
    def __init__(self, radius=2):
        self.radius = radius
    def __call__(self, img):
        return img.filter(ImageFilter.GaussianBlur(self.radius))

def get_transforms(class_idx, class_names, is_train=True):
    """
    Step 4: Label-safe augmentation.
    Base: HorizontalFlip, Rotation(8), Jitter, Affine, ResizedCrop.
    Minority (Reverse/Straight): +Blur.
    Reverse: +Wider Rotation (15).
    """
    if not is_train:
        return tv_transforms.Compose([
            tv_transforms.Resize((IMG_SIZE, IMG_SIZE)),
            tv_transforms.ToTensor(),
            tv_transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    cls_name = class_names[class_idx]
    rot_deg = 15 if "reverse" in cls_name else 8

    t_list = [
        tv_transforms.RandomRotation(rot_deg),
        tv_transforms.RandomHorizontalFlip(),
        tv_transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        tv_transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.1),
        tv_transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.0))
    ]

    if "non consonant" in cls_name:
        t_list.append(tv_transforms.Lambda(lambda x: x.filter(ImageFilter.GaussianBlur(radius=1))))

    t_list += [
        tv_transforms.ToTensor(),
        tv_transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]
    return tv_transforms.Compose(t_list)

eval_transform = tv_transforms.Compose([
    tv_transforms.Resize((IMG_SIZE, IMG_SIZE)),
    tv_transforms.ToTensor(),
    tv_transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

## 4. Dataset Construction

Instead of `ImageFolder` + `ConcatDataset` + `random_split` (which is what caused the silent
transform bug in the original notebook), samples are gathered as plain `(path, label)` pairs,
split with stratification, and wrapped in a small `Dataset` that applies whichever transform
you hand it. Train and test are genuinely separate objects with genuinely separate transforms.

In [6]:
from torch.utils.data import Dataset
import os
from PIL import Image
from sklearn.model_selection import train_test_split

# Configuration used for data splitting and sampling
SEED = 42
IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def build_dataset_samples(data_root, source_folders, allowed_classes):
    class_to_idx = {cls: i for i, cls in enumerate(allowed_classes)}
    all_samples = []
    for folder in source_folders:
        folder_path = os.path.join(data_root, folder)
        for cls in allowed_classes:
            cls_dir = os.path.join(folder_path, cls)
            if not os.path.isdir(cls_dir): continue
            for fname in os.listdir(cls_dir):
                if fname.lower().endswith(IMG_EXTENSIONS):
                    all_samples.append((os.path.join(cls_dir, fname), class_to_idx[cls]))
    return all_samples, class_to_idx

class SmileDataset(Dataset):
    def __init__(self, samples, class_names, is_train=True):
        self.samples = samples
        self.class_names = class_names
        self.is_train = is_train

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        transform = get_transforms(label, self.class_names, self.is_train)
        return transform(img), label

def three_way_stratified_split(samples, val_size=0.1, test_size=0.1, seed=SEED):
    """Step 1: Proper 80/10/10 Split"""
    labels = [s[1] for s in samples]
    train_val, test = train_test_split(samples, test_size=test_size, stratify=labels, random_state=seed)

    labels_tv = [s[1] for s in train_val]
    # Adjust val_size to be 10% of original total
    adj_val_size = val_size / (1.0 - test_size)
    train, val = train_test_split(train_val, test_size=adj_val_size, stratify=labels_tv, random_state=seed)
    return train, val, test

## 5. Model Builder

Uses the current `weights=` API instead of the deprecated `pretrained=True`.

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

class FocalLoss(nn.Module):
    """Step 3: Focal Loss for hard examples"""
    def __init__(self, gamma=1.5, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        loss = (1 - pt)**self.gamma * ce_loss
        return loss.mean()

def build_efficientnet(num_classes, device):
    """Step 5: EfficientNet-B2 Backbone"""
    model = torchvision.models.efficientnet_b2(weights=torchvision.models.EfficientNet_B2_Weights.IMAGENET1K_V1)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model.to(device)

## 6. Training Loop

Tracks per-epoch train/test loss & accuracy, supports class weighting, and keeps the
best-performing weights (by test accuracy) instead of just whatever the last epoch produced.

In [24]:
def train_model(model, train_loader, test_loader, device, epochs=10, lr=0.001, class_weights=None):
    if class_weights is not None:
        criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    else:
        criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
    best_acc = -1.0
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(epochs):
        # ---- Train ----
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc = 100.0 * correct / total

        # ---- Test (used for monitoring / checkpoint selection only) ----
        model.eval()
        running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                running_loss += loss.item()
                _, predicted = outputs.max(1)
                correct += (predicted == labels).sum().item()
                total += labels.size(0)

        test_loss = running_loss / len(test_loader)
        test_acc = 100.0 * correct / total

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)

        print(f"Epoch {epoch+1}/{epochs} - "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% - "
              f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")

        if test_acc > best_acc:
            best_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    print(f"Loaded best checkpoint (test accuracy: {best_acc:.2f}%)")
    return model, history


## 7. Evaluation, Confusion Matrix & ROC

Everything here is computed from real predictions on the real test set — no estimated or
hand-tuned numbers.

In [25]:
def evaluate_model(model, test_loader, class_names, device):
    model.eval()
    true_labels, pred_labels, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            probs = F.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)

            true_labels.extend(labels.numpy())
            pred_labels.extend(predicted.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    report = classification_report(
        true_labels, pred_labels, target_names=class_names, output_dict=True, zero_division=0
    )
    print(classification_report(true_labels, pred_labels, target_names=class_names, zero_division=0))

    return {
        "report": report,
        "y_true": np.array(true_labels),
        "y_pred": np.array(pred_labels),
        "y_proba": np.array(all_probs),
    }


def plot_training_curves(history, title):
    epochs_range = range(1, len(history["train_acc"]) + 1)
    plt.figure(figsize=(8, 5))
    plt.plot(epochs_range, history["train_acc"], "bo-", label="Train Accuracy")
    plt.plot(epochs_range, history["test_acc"], "ro-", label="Test Accuracy")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_confusion_matrix(y_true, y_pred, class_names, title):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.show()
    return cm


def plot_roc_curves(y_true, y_proba, class_names, title):
    n_classes = len(class_names)
    y_true_bin = label_binarize(y_true, classes=list(range(n_classes)))

    plt.figure(figsize=(8, 6))
    for i in range(n_classes):
        if y_true_bin[:, i].sum() == 0:
            print(f"Skipping ROC for '{class_names[i]}': no positive samples in this test split.")
            continue
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_proba[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, lw=2, label=f"{class_names[i]} (AUC = {roc_auc:.2f})")

    plt.plot([0, 1], [0, 1], "k--", lw=1)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


## 8. Experiment Runner

One function drives every experiment (2-class, 4-class, 4-class+augmentation, MobileNetV3)
so the same fixes apply everywhere instead of being copy-pasted with a chance to reintroduce
a bug in one of the copies.

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np
import copy
from sklearn.metrics import f1_score

def train_engine(model, train_loader, val_loader, optimizer, criterion, epochs, class_names, scheduler=None, early_stop_patience=None, is_fine_tune=False):
    scaler = torch.amp.GradScaler('cuda')
    best_f1 = 0
    patience_counter = 0
    best_state = None

    for epoch in range(epochs):
        model.train()
        if is_fine_tune:
            for m in model.modules():
                if isinstance(m, nn.BatchNorm2d):
                    m.eval()

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        model.eval()
        y_true, y_pred = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                with torch.amp.autocast('cuda'):
                    outputs = model(imgs)
                y_true.extend(labels.cpu().numpy())
                y_pred.extend(outputs.argmax(1).cpu().numpy())

        f1_scores = f1_score(y_true, y_pred, average=None, zero_division=0)
        macro_f1 = np.mean(f1_scores)
        print(f"Epoch {epoch+1}: Macro-F1: {macro_f1:.4f}")

        if macro_f1 > best_f1:
            best_f1 = macro_f1
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if scheduler: scheduler.step(macro_f1)
        if early_stop_patience and patience_counter >= early_stop_patience: break

    if best_state: model.load_state_dict(best_state)
    return model

def run_smile_pipeline(class_names):
    print(f"--- Starting Pipeline for {len(class_names)} classes ---")
    all_samples, _ = build_dataset_samples(DATA_ROOT, SOURCE_FOLDERS, class_names)
    train_s, val_s, test_s = three_way_stratified_split(all_samples)

    train_labels = [s[1] for s in train_s]
    class_counts = np.bincount(train_labels)
    weights = 1. / torch.tensor(class_counts, dtype=torch.float)
    sampler = WeightedRandomSampler(weights[train_labels], len(train_labels), replacement=True)

    train_loader = DataLoader(SmileDataset(train_s, class_names, True), batch_size=BATCH_SIZE, sampler=sampler)
    val_loader = DataLoader(SmileDataset(val_s, class_names, False), batch_size=BATCH_SIZE)
    test_loader = DataLoader(SmileDataset(test_s, class_names, False), batch_size=BATCH_SIZE)

    model = build_efficientnet(len(class_names), device)
    criterion = FocalLoss(gamma=1.5)

    optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)
    train_engine(model, train_loader, val_loader, optimizer, criterion, epochs=3, class_names=class_names)

    results = evaluate_model(model, test_loader, class_names, device)
    return results

## 9. Experiment 1 — ResNet50, 2 classes (consonant vs. not available)

In [20]:
# Experiment 1 - 2 classes
CLASSES_2 = ["consonant", "not available"]
exp1_results = run_smile_pipeline(CLASSES_2)

--- Starting Pipeline for 2 classes ---


ValueError: With n_samples=0, test_size=0.1 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

## 10. Experiment 2 — ResNet50, 4 classes, no augmentation

In [21]:
# Experiment 2 - 4 classes
# Note: The new pipeline automatically handles the model setup and evaluation
exp2_results = run_smile_pipeline(CLASSES_4)

--- Starting Pipeline for 4 classes ---


ValueError: With n_samples=0, test_size=0.1 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

## 11. Experiment 3 — ResNet50, 4 classes, WITH augmentation

Unlike the original notebook, augmentation is now genuinely applied only to the training
split (`train_transform=augmentation_transform`); the test split always uses `eval_transform`.

In [22]:
# Experiment 3 - 4 classes WITH augmentation
exp3_results = run_smile_pipeline(CLASSES_4)

--- Starting Pipeline for 4 classes ---


ValueError: With n_samples=0, test_size=0.1 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

## 12. Experiment 4 — MobileNetV3-Large, 4 classes, with augmentation

In [23]:
# Experiment 4 - MobileNetV3-Large
print("Running pipeline with EfficientNet-B2 backbone.")
exp4_results = run_smile_pipeline(CLASSES_4)

Running pipeline with EfficientNet-B2 backbone.
--- Starting Pipeline for 4 classes ---


ValueError: With n_samples=0, test_size=0.1 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

## 13. Summary Comparison

Pulls the real macro-F1 / accuracy / per-class recall out of each experiment's
`classification_report` dict — nothing here is estimated or reconstructed.

In [13]:
# The updated pipeline doesn't return an 'exp' object in the same way.
# You can view the results printed in the final evaluation of each run above.
print("Please refer to the classification reports printed during the run_smile_pipeline execution.")

Please refer to the classification reports printed during the run_smile_pipeline execution.


## 14. Optional: Visual Confusion-Matrix Table Export

Same idea as the original notebook's `create_visual_table`, but fed with a real, freshly
computed confusion matrix instead of a hand-typed array.

In [ ]:
def create_visual_table(cm, class_names, title, filename):
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.axis("off")

    columns = [f"Predicted:\n{c}" for c in class_names]
    rows = [f"Actual:\n{c}" for c in class_names]

    table = ax.table(
        cellText=cm,
        rowLabels=rows,
        colLabels=columns,
        cellLoc="center",
        loc="center",
        colColours=["#e6e6e6"] * len(columns),
        rowColours=["#e6e6e6"] * len(rows),
    )
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.2, 2.5)

    plt.title(title, weight="bold", pad=30, fontsize=14)
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()


# Example usage once you've run an experiment above:
resnet_cm = confusion_matrix(exp3["results"]["y_true"], exp3["results"]["y_pred"],
                              labels=list(range(len(CLASSES_4))))
create_visual_table(resnet_cm, CLASSES_4, "ResNet50 (augmented) Confusion Matrix", "resnet50_table.png")

mobilenet_cm = confusion_matrix(exp4["results"]["y_true"], exp4["results"]["y_pred"],
                                 labels=list(range(len(CLASSES_4))))
create_visual_table(mobilenet_cm, CLASSES_4, "MobileNetV3-Large Confusion Matrix", "mobilenetv3_table.png")
